# TelecomX BR — Modelagem de Churn com scikit-learn

Notebook de referência para construção de baseline preditivo com pipeline reproduzível:
- ingestão robusta
- limpeza e padronização
- pré-processamento com `ColumnTransformer`
- modelos baseline (Regressão Logística e Random Forest)
- métricas de classificação e ajuste de threshold


In [ ]:
import json
import requests
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    recall_score,
    precision_score,
    confusion_matrix,
    classification_report,
)

pd.set_option('display.max_columns', None)
RANDOM_STATE = 42


In [ ]:
URL_DADOS = 'https://raw.githubusercontent.com/ingridcristh/challenge2-data-science/main/TelecomX_Data.json'

response = requests.get(URL_DADOS, timeout=30)
response.raise_for_status()

raw_data = json.loads(response.text)
df = pd.json_normalize(raw_data, sep='_')

print(f'Linhas: {df.shape[0]} | Colunas: {df.shape[1]}')
df.head(3)


In [ ]:
# Padronização de colunas

df.columns = df.columns.str.lower()

rename_map = {
    'customerid': 'id_cliente',
    'churn': 'churn',
    'customer_gender': 'genero',
    'customer_seniorcitizen': 'cidadao_senior',
    'customer_partner': 'parceiro',
    'customer_dependents': 'dependentes',
    'customer_tenure': 'meses_contrato',
    'phone_phoneservice': 'servico_telefone',
    'phone_multiplelines': 'multiplas_linhas',
    'internet_internetservice': 'servico_internet',
    'internet_onlinesecurity': 'seguranca_online',
    'internet_onlinebackup': 'backup_online',
    'internet_deviceprotection': 'protecao_dispositivo',
    'internet_techsupport': 'suporte_tecnico',
    'internet_streamingtv': 'streaming_tv',
    'internet_streamingmovies': 'streaming_filmes',
    'account_contract': 'contrato',
    'account_paperlessbilling': 'fatura_online',
    'account_paymentmethod': 'metodo_pagamento',
    'account_charges_monthly': 'gastos_mensais',
    'account_charges_total': 'gastos_totais',
}

missing_in_rename = sorted(set(rename_map).difference(df.columns))
if missing_in_rename:
    raise ValueError(f'Chaves de rename ausentes no dataframe: {missing_in_rename}')

df = df.rename(columns=rename_map)

# Limpeza de id
if 'id_cliente' in df.columns:
    df['id_cliente'] = df['id_cliente'].astype(str).str.replace('-', '', regex=False)

df.head(3)


In [ ]:
# Conversões de tipo e tratamento de missing

df['gastos_totais'] = pd.to_numeric(df['gastos_totais'], errors='coerce')

# NaN em gastos_totais geralmente ocorre quando meses_contrato = 0.
# Estratégia: imputação mediana para manter o volume da base.

# Feature engineering

df['contas_diarias'] = df['gastos_mensais'] / 30

df['gasto_medio_mensal_por_tempo'] = np.where(
    df['meses_contrato'] > 0,
    df['gastos_totais'] / df['meses_contrato'],
    np.nan,
)

df['contrato_mensal'] = (df['contrato'] == 'Month-to-month').astype(int)
df['pagamento_eletronico'] = (df['metodo_pagamento'] == 'Electronic check').astype(int)
df['fibra'] = (df['servico_internet'] == 'Fiber optic').astype(int)

# Alvo binário

df['target_churn'] = (df['churn'] == 'Yes').astype(int)

df[['churn', 'target_churn', 'gastos_totais', 'gasto_medio_mensal_por_tempo']].head()


In [ ]:
# Definição de variáveis para modelagem

colunas_excluir = ['churn', 'id_cliente']
X = df.drop(columns=colunas_excluir + ['target_churn'])
y = df['target_churn']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

num_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X_train.select_dtypes(exclude=['int64', 'float64']).columns.tolist()

print(f'Treino: {X_train.shape}, Teste: {X_test.shape}')
print(f'Numéricas: {len(num_features)} | Categóricas: {len(cat_features)}')


In [ ]:
preprocessador = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline(
                steps=[
                    ('imputer', SimpleImputer(strategy='median')),
                    ('scaler', StandardScaler()),
                ]
            ),
            num_features,
        ),
        (
            'cat',
            Pipeline(
                steps=[
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    ('onehot', OneHotEncoder(handle_unknown='ignore')),
                ]
            ),
            cat_features,
        ),
    ]
)

modelos = {
    'logistic_regression': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    'random_forest': RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=2,
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}


In [ ]:
# Cross-validation apenas no treino
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    'roc_auc': 'roc_auc',
    'average_precision': 'average_precision',
    'f1': 'f1',
    'recall': 'recall',
    'precision': 'precision',
}

resultados_cv = []

for nome, modelo in modelos.items():
    pipe = Pipeline(steps=[('preprocessador', preprocessador), ('modelo', modelo)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

    resultados_cv.append({
        'modelo': nome,
        **{metrica: scores[f'test_{metrica}'].mean() for metrica in scoring},
    })

cv_df = pd.DataFrame(resultados_cv).sort_values('roc_auc', ascending=False)
cv_df


In [ ]:
# Treino final e avaliação no teste (holdout)

def avaliar_modelo(pipe, X_treino, y_treino, X_teste, y_teste, threshold=0.5):
    pipe.fit(X_treino, y_treino)
    probas = pipe.predict_proba(X_teste)[:, 1]
    preds = (probas >= threshold).astype(int)

    return {
        'roc_auc': roc_auc_score(y_teste, probas),
        'pr_auc': average_precision_score(y_teste, probas),
        'f1': f1_score(y_teste, preds),
        'recall': recall_score(y_teste, preds),
        'precision': precision_score(y_teste, preds),
        'threshold': threshold,
        'confusion_matrix': confusion_matrix(y_teste, preds),
        'classification_report': classification_report(y_teste, preds, digits=4),
        'probas': probas,
    }

modelos_treinados = {}
metricas_teste = []

for nome, modelo in modelos.items():
    pipe = Pipeline(steps=[('preprocessador', preprocessador), ('modelo', modelo)])
    resultado = avaliar_modelo(pipe, X_train, y_train, X_test, y_test, threshold=0.5)
    modelos_treinados[nome] = (pipe, resultado)

    metricas_teste.append({
        'modelo': nome,
        'roc_auc': resultado['roc_auc'],
        'pr_auc': resultado['pr_auc'],
        'f1': resultado['f1'],
        'recall': resultado['recall'],
        'precision': resultado['precision'],
    })

pd.DataFrame(metricas_teste).sort_values('roc_auc', ascending=False)


In [ ]:
# Ajuste simples de threshold para priorizar recall da classe churn (exemplo com melhor ROC-AUC)
melhor_modelo = pd.DataFrame(metricas_teste).sort_values('roc_auc', ascending=False).iloc[0]['modelo']
pipe_melhor, _ = modelos_treinados[melhor_modelo]

thresholds = np.arange(0.25, 0.71, 0.05)
linhas = []

for t in thresholds:
    r = avaliar_modelo(pipe_melhor, X_train, y_train, X_test, y_test, threshold=float(t))
    linhas.append({'threshold': t, 'f1': r['f1'], 'recall': r['recall'], 'precision': r['precision']})

threshold_df = pd.DataFrame(linhas)
threshold_df


In [ ]:
# Escolha de threshold por melhor F1 (poderia ser por maior recall, conforme estratégia de retenção)
best_threshold = threshold_df.sort_values('f1', ascending=False).iloc[0]['threshold']
resultado_final = avaliar_modelo(pipe_melhor, X_train, y_train, X_test, y_test, threshold=float(best_threshold))

print(f'Modelo escolhido: {melhor_modelo}')
print(f'Threshold escolhido: {best_threshold:.2f}')
print('Matriz de confusão:')
print(resultado_final['confusion_matrix'])
print('Relatório de classificação:')
print(resultado_final['classification_report'])


## Próximos passos recomendados
1. Testar modelos com calibração (`CalibratedClassifierCV`) para melhorar probabilidades.
2. Avaliar estabilidade temporal (se houver coluna de data em versões futuras dos dados).
3. Registrar experimento (MLflow/W&B) e exportar pipeline com `joblib`.
